# Sci-Fi Genre Timeline

Tracks how sci-fi sub-genres rise and fall over decades using the
`scifi_novels` and `scifi_films` tables.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine

REPO    = Path("../").resolve()
DB_PATH = REPO / "data/databases/scifi.db"
engine  = create_engine(f"sqlite:///{DB_PATH}")

novels  = pd.read_sql_table("scifi_novels", engine)
films   = pd.read_sql_table("scifi_films",  engine)
engine.dispose()

print(f"novels={len(novels):,}  films={len(films):,}")

novels=6,306  films=4,204


## 1  Output volume by decade

In [ ]:
novels["decade"] = (novels["pub_year"].dropna().astype(int) // 10 * 10)
films["decade"]  = (films["release_year"].dropna().astype(int) // 10 * 10)

vol = pd.DataFrame({
    "Novels": novels.groupby("decade").size(),
    "Films":  films.groupby("decade").size(),
}).loc[1900:2030].fillna(0).astype(int)

fig = px.bar(
    vol.reset_index().melt(id_vars="decade", var_name="type", value_name="count"),
    x="decade", y="count", color="type", barmode="group",
    title="Sci-fi novel and film output by decade",
    labels={"decade": "Decade", "count": "Wikipedia pages"},
    width=1000, height=450,
)
fig.show()

## 2  Genre × decade heatmap (novels)

In [ ]:
# Explode pipe-separated genre strings
novel_genres = (
    novels[["pub_year", "genres"]]
    .dropna()
    .assign(decade=lambda d: (d["pub_year"].astype(int) // 10 * 10))
    .assign(genre=lambda d: d["genres"].str.split("|"))
    .explode("genre")
    .assign(genre=lambda d: d["genre"].str.strip().str.lower())
    .query("genre != '' and 1940 <= decade <= 2020")
)

# Top genres by total count
top_genres = novel_genres["genre"].value_counts().head(20).index.tolist()
pivot = (
    novel_genres[novel_genres["genre"].isin(top_genres)]
    .groupby(["genre", "decade"])
    .size()
    .unstack(fill_value=0)
    .loc[top_genres]
)

# Normalise each genre to its own max so rare genres are still visible
pivot_norm = pivot.div(pivot.max(axis=1).replace(0, 1), axis=0)

fig = px.imshow(
    pivot_norm,
    aspect="auto",
    color_continuous_scale="Blues",
    title="Novel genre popularity by decade (normalised to each genre's peak)",
    labels={"x": "Decade", "y": "Genre", "color": "Relative volume"},
    width=1000, height=600,
)
fig.show()

## 3  Genre rise / fall (stacked area — novels)

In [ ]:
TOP_N_AREA = 10
top_area_genres = novel_genres["genre"].value_counts().head(TOP_N_AREA).index.tolist()

area_df = (
    novel_genres[novel_genres["genre"].isin(top_area_genres)]
    .groupby(["decade", "genre"])
    .size()
    .reset_index(name="count")
)

fig = px.area(
    area_df, x="decade", y="count", color="genre",
    title=f"Top {TOP_N_AREA} novel genres over time (stacked area)",
    labels={"decade": "Decade", "count": "Wikipedia pages", "genre": "Genre"},
    width=1000, height=500,
)
fig.show()

## 4  Film output by decade — top directors over time

In [ ]:
film_dirs = (
    films[["release_year", "director"]]
    .dropna()
    .assign(decade=lambda d: (d["release_year"].astype(int) // 10 * 10))
    .assign(director=lambda d: d["director"].str.split("|"))
    .explode("director")
    .assign(director=lambda d: d["director"].str.strip())
    .query("director != '' and 1950 <= decade <= 2020")
)

top_dirs = film_dirs["director"].value_counts().head(15).index.tolist()

dir_pivot = (
    film_dirs[film_dirs["director"].isin(top_dirs)]
    .groupby(["decade", "director"])
    .size()
    .reset_index(name="films")
)

fig = px.bar(
    dir_pivot, x="decade", y="films", color="director", barmode="stack",
    title="Films by top 15 sci-fi directors over time",
    width=1000, height=500,
)
fig.show()

## 5  Adaptation lag — novel pub year vs film release year

For films with a `based_on` field, try to find the source novel in the novels table
and measure how many years elapsed before the film was made.

In [ ]:
adapted_films = films.dropna(subset=["based_on", "release_year"]).copy()

# Fuzzy match: find novels whose title appears in the based_on string
novel_lookup = novels.dropna(subset=["pub_year", "title"]).set_index("title")["pub_year"]

lags = []
for _, row in adapted_films.iterrows():
    based = str(row["based_on"]).lower()
    for title, pub_year in novel_lookup.items():
        if len(title) > 4 and title.lower() in based:
            lag = int(row["release_year"]) - int(pub_year)
            if 0 <= lag <= 80:
                lags.append({"film": row["title"], "novel": title,
                             "pub_year": int(pub_year),
                             "release_year": int(row["release_year"]),
                             "lag_years": lag})
            break

lag_df = pd.DataFrame(lags).sort_values("lag_years")
print(f"Matched {len(lag_df):,} novel→film adaptations")

if len(lag_df) > 5:
    fig = px.histogram(
        lag_df, x="lag_years", nbins=30,
        title="Years from novel publication to film release",
        labels={"lag_years": "Lag (years)"},
        width=800, height=400,
    )
    fig.show()
    display(lag_df.head(20))

Matched 7 novel→film adaptations


,film,novel,pub_year,release_year,lag_years
5,Maze Runner (film series),The Maze Runner,2009,2014,5
6,God's Puzzle (film),God's Puzzle,2002,2008,6
4,Memoirs of a Survivor (film),The Memoirs of a Survivor,1974,1981,7
0,Donovan's Brain (film),Donovan's Brain,1942,1953,11
3,Alraune (1952 film),Alraune,1911,1952,41
1,The Puppet Masters (film),The Puppet Masters,1951,1994,43
2,Abbott and Costello Meet Dr. Jekyll and Mr. Hyde,Strange Case of Dr Jekyll and Mr Hyde,1886,1953,67
